# Preprocessing

In [1]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
import pandas as pd
import similarity.preprocessing as pp

raw = pp.load_raw()
raw.shape

(30000, 13)

In [3]:
raw.head(3)

,uniq_id,asin,product_name,brand,colour,sales_price,weight,rating,no__of_reviews,product_details__k_v_pairs,sales_rank_in_child_category,browsenode,medium
0,26d41bdc1495de290bc8e6062d927729,B07STS2W9T,LA' Facon Cotton Kalamkari Handblock Saree Blo...,LA' Facon,NaN,200.0,999999999,5.0,NaN,"{'Item_part_number': 'Devi face', 'ASIN': 'B07...",{'WomensKurtasKurtis': '#1793'},1.968255e+09,https://images-na.ssl-images-amazon.com/images...
1,410c62298852e68f34c35560f2311e5a,B07N6TD2WL,Sf Jeans By Pantaloons Men's Plain Slim fit T-...,NaN,NaN,265.0,999999999,3.6,NaN,"{'ASIN': 'B07N6TD2WL', 'Date_first_available_a...",{'MensT_Shirts': '#12151'},1.968123e+09,https://images-na.ssl-images-amazon.com/images...
2,52e31bb31680b0ec73de0d781a23cc0a,B07WJ6WPN1,LOVISTA Cotton Gota Patti Tassel Traditional P...,LOVISTA,NaN,660.0,999999999,3.5,NaN,"{'Item_part_number': 'KUM-011', 'ASIN': 'B07WJ...",{'WomensKurtasKurtis': '#91'},1.968255e+09,https://images-na.ssl-images-amazon.com/images...


## brand

In [4]:
brand = raw["brand"].head(10)
pd.DataFrame({"raw": brand, "clean": brand.map(pp.parse_brand)})

,raw,clean
0,LA' Facon,la' facon
1,NaN,None
2,LOVISTA,lovista
3,NaN,None
4,NaN,None
5,Forest Club,forest club
6,PrintOctopus,printoctopus
7,NaN,None
8,Carahere,carahere
9,Peppermint,peppermint


In [5]:
pp.parse_brand("Generic"), pp.parse_brand("N/A")

(None, None)

## colour

In [6]:
colour = raw["colour"].dropna().head(10)
pd.DataFrame({"raw": colour, "clean": colour.map(pp.parse_colours)})

,raw,clean
5,Black|Blue|Grey,"[black, blue, grey]"
6,Black|Navy Blue,"[black, navy blue]"
15,Black|Maroon,"[black, maroon]"
20,Black Based|Black Based 2|Black White|Blue Whi...,"[black based, black based 2, black white, blue..."
22,Black|Bottle green|GREEN|NAVY BLUE|Pink|Red|WH...,"[black, blackk, blue, bottle green, gray, gree..."
23,Black|White,"[black, white]"
29,Black|Blue,"[black, blue]"
33,Black|Pink|Yellow,"[black, pink, yellow]"
34,Green|Pink|Red,"[green, pink, red]"
36,Black Half|Black Sando|Rust|Black Full|Maroon|...,"[black full, black half, black sando, maroon, ..."


## weight

In [7]:
pattern = raw["weight"].str.replace(r"[\d.]+", "N", regex=True)
examples = raw.groupby(pattern)["weight"].first()
pd.DataFrame({"raw": examples, "grams": examples.map(pp.parse_weight_grams)})

,raw,grams
weight,,
N,999999999,NaN
N Grams,454 Grams,454.0
N Kg,1.4 Kg,1400.0
N Kilograms,113 Kilograms,NaN
N g,99.8 g,99.8
N grams,975 grams,975.0
N grams,458.0 grams,458.0
The Slim N Lift slimming shirt even smoothest out your back for a lean tapered appearanceN Now you can look inches smaller and pound thinner without exercise,The Slim N Lift slimming shirt even smoothest ...,NaN


In [8]:
pp.parse_weight_grams("113 Kilograms"), pp.parse_weight_grams("4.54 g")

(None, None)

In [9]:
raw["weight"].map(pp.parse_weight_grams).describe()

count    6184.000000
mean      257.845959
std       180.323286
min         9.070000
25%       150.000000
50%       240.000000
75%       349.000000
max      4540.000000
Name: weight, dtype: float64

## price and rating

In [10]:
raw["sales_price"].map(pp.parse_price).describe()

count    27110.000000
mean       862.172397
std        964.223008
min         39.000000
25%        379.000000
50%        590.000000
75%        899.000000
max       9988.000000
Name: sales_price, dtype: float64

In [11]:
raw["rating"].map(pp.parse_rating).describe()

count    30000.000000
mean         4.039857
std          0.840009
min          1.000000
25%          3.500000
50%          4.000000
75%          4.900000
max          5.000000
Name: rating, dtype: float64

## reviews

In [12]:
reviews = pd.Series(
    [pp.parse_review_count(d, c) for d, c in zip(raw["product_details__k_v_pairs"], raw["no__of_reviews"])]
)
reviews.describe()

count    29483.000000
mean        24.486043
std        188.192838
min          0.000000
25%          1.000000
50%          2.000000
75%          7.000000
max       9896.000000
dtype: float64

In [13]:
reviews.isna().sum(), (reviews == 0).sum()

(np.int64(517), np.int64(3892))

## category, browsenode, image

In [14]:
raw["sales_rank_in_child_category"].head(5).map(pp.parse_category)

0    WomensKurtasKurtis
1          MensT_Shirts
2    WomensKurtasKurtis
3          MensT_Shirts
4        MensTracksuits
Name: sales_rank_in_child_category, dtype: object

In [15]:
raw["browsenode"].head(5).map(pp.parse_browsenode)

0    1968255031
1    1968123031
2    1968255031
3    1968123031
4    1968070031
Name: browsenode, dtype: object

In [16]:
raw["medium"].head(3).map(pp.parse_first_url)

0    https://images-na.ssl-images-amazon.com/images...
1    https://images-na.ssl-images-amazon.com/images...
2    https://images-na.ssl-images-amazon.com/images...
Name: medium, dtype: object

## build

In [17]:
products = pp.build_products(raw)
products.shape

(30000, 13)

In [18]:
products.head()

,uniq_id,asin,name,brand,colours,price,weight_g,rating,review_count,child_category,browsenode,image_url,rating_adjusted
0,26d41bdc1495de290bc8e6062d927729,B07STS2W9T,LA' Facon Cotton Kalamkari Handblock Saree Blo...,la' facon,None,200.0,NaN,5.0,3.0,WomensKurtasKurtis,1968255031,https://images-na.ssl-images-amazon.com/images...,4.484605
1,410c62298852e68f34c35560f2311e5a,B07N6TD2WL,Sf Jeans By Pantaloons Men's Plain Slim fit T-...,None,None,265.0,NaN,3.6,3.0,MensT_Shirts,1968123031,https://images-na.ssl-images-amazon.com/images...,3.959605
2,52e31bb31680b0ec73de0d781a23cc0a,B07WJ6WPN1,LOVISTA Cotton Gota Patti Tassel Traditional P...,lovista,None,660.0,NaN,3.5,23.0,WomensKurtasKurtis,1968255031,https://images-na.ssl-images-amazon.com/images...,3.620601
3,25798d6dc43239c118452d1bee0fb088,B07PYSF4WZ,People Men's Printed Regular fit T-Shirt,None,None,195.0,NaN,3.0,1.0,MensT_Shirts,1968123031,https://images-na.ssl-images-amazon.com/images...,3.979473
4,ad8a5a196d515ef09dfdaf082bdc37c4,B082KXNM7X,Monte Carlo Grey Solid Cotton Blend Polo Colla...,None,None,1914.0,NaN,5.0,0.0,MensTracksuits,1968070031,https://images-na.ssl-images-amazon.com/images...,4.175367


In [19]:
products.dtypes

uniq_id             object
asin                object
name                object
brand               object
colours             object
price              float64
weight_g           float64
rating             float64
review_count       float64
child_category      object
browsenode          object
image_url           object
rating_adjusted    float64
dtype: object

In [20]:
pp.fill_summary(products)

,filled,percent
uniq_id,30000,100.0
asin,30000,100.0
name,30000,100.0
brand,21608,72.0
colours,6029,20.1
price,27110,90.4
weight_g,6184,20.6
rating,30000,100.0
review_count,29483,98.3
child_category,24851,82.8


In [21]:
products["asin"].duplicated(keep=False).sum()

np.int64(922)

## rating_adjusted

In [22]:
products.loc[products["review_count"] == 1, ["rating", "review_count", "rating_adjusted"]].head()

,rating,review_count,rating_adjusted
3,3.0,1.0,3.979473
9,5.0,1.0,4.312806
12,2.5,1.0,3.896139
13,5.0,1.0,4.312806
19,5.0,1.0,4.312806


In [23]:
products.sort_values("rating_adjusted", ascending=False)[["name", "rating", "review_count", "rating_adjusted"]].head(10)

,name,rating,review_count,rating_adjusted
7528,"Sally Hansen 2 Count, Big Matte + Big Kwik Tre...",5.0,1154.0,4.996442
22183,Columbia Men's Tamiami II Short Sleeve Fishing...,5.0,922.0,4.995552
4032,Epoch Mens Premium Wool Blend Classic Flat Ivy...,5.0,573.0,4.992866
7603,Gerber Plastic Nylon Pants Fits Up to 12 lbs (...,5.0,532.0,4.992322
2230,Columbia Sportswear Men's Perfect Cast Polo Sh...,5.0,470.0,4.991320
28719,"Mueller Tennis Elbow W/Gel Pad, One Size Fits ...",5.0,400.0,4.989819
23713,Teal and Orange Striped Woven Silk Extra Long Tie,5.0,370.0,4.989005
20269,Hatsandscarf CC Exclusives Unisex Solid Ribbed...,5.0,331.0,4.987729
556,"Hanes Tagless T-Shirt 5250, Kelly Green, XXX-L...",5.0,324.0,4.987468
26652,Pensee Mens Self Bow Tie Multi-color Plaids & ...,5.0,246.0,4.983573


## save

In [24]:
pp.validate(products)

In [25]:
pp.save(products)

WindowsPath('data/processed/products.parquet')

In [26]:
pp.load_products().shape

(30000, 13)